# Stratify existing results by `concept_type`

Companion to `TODO.md` item 2: `mapped_concepts.parquet` has a `concept_type` column
(`SCT_PRE` / `SCT_POST`) that nothing in the pipeline uses. This notebook breaks the
`eval.py` metrics down by it, to answer a reviewer-obvious question: does a candidate
list's advantage over a baseline hold uniformly across concept types, or is it
concentrated in a few?

`SCT_PRE` = plain, pre-coordinated SNOMED concept ids (e.g. `224366004`). `SCT_POST` =
compound, post-coordinated expressions (e.g. `313696009:116686009=122554006`) --
confirmed by inspecting `mapped_concepts.parquet` directly, not documented elsewhere in
this repo.

**What you can change** (see "Parameters" below): which candidate lists to compare
(`CANDIDATE_LISTS`, defaults to the best-performing list vs. the strongest baseline per
`TODO.md`), and which `k` value(s) to stratify (`K_LIST`, doesn't need the full sweep).

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import pickle

import matplotlib.pyplot as plt
import polars as pl

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.eval as eval
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
from src.graph_tokenizer_gd_tree_dev import drilldown

## Load graph, mapped concepts (+ `concept_type`), and available candidate lists

In [3]:
with open(config.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)
with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

df_mapped = pl.read_parquet(config.BasicConfig().mapped_path)
mapped_ids = sorted(df_mapped["id"].unique().to_list())
D = config.TokenizerParam().max_dist_candidate

# concept_type is a per-id property (every id has exactly one), but mapped_concepts.parquet
# has multiple rows per id (one per source pointer) -- .unique() so the later join can't fan out.
concept_type = df_mapped.select("id", "concept_type").unique()
print(concept_type.group_by("concept_type").len().sort("concept_type"))

all_candidates = drilldown.load_all_candidates()
adj = tokenizer.build_out_adjacency(combined_subgraphs)
A, node_to_idx = tokenizer.build_coverage_transition(combined_subgraphs)

print("Available candidate lists -- (category, file_type) pairs to use in CANDIDATE_LISTS below:")
for category, file_types in all_candidates.items():
    print(f"  {category}: {sorted(file_types)}")

shape: (2, 2)
┌──────────────┬───────┐
│ concept_type ┆ len   │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ SCT_POST     ┆ 26311 │
│ SCT_PRE      ┆ 19839 │
└──────────────┴───────┘
Available candidate lists -- (category, file_type) pairs to use in CANDIDATE_LISTS below:
  greedy_tree_margin: ['0.0', '0.2', '0.4', '0.6', '0.8', '1.0']
  baseline: ['highest_degree', 'highest_degree_dist_1', 'k_random_all_samples', 'most_children']


## Parameters

Edit these, then re-run every cell below.

In [4]:
CANDIDATE_LISTS = [
    ("greedy_tree_margin", "0.8"),
    ("baseline", "highest_degree_dist_1"),
]  # per TODO.md item 2: the best-performing list vs. the strongest baseline by default.
# The "gap" cell further below specifically compares CANDIDATE_LISTS[0] against
# CANDIDATE_LISTS[1] -- add more entries if you want extra methods in the tables/plots, but
# the headline gap stays keyed to the first two.

K_LIST = [1000, 5000, 8000]  # one or two k values, per TODO.md item 2 -- doesn't need the full sweep.

## Per-concept metrics for each `(method, k)`, joined to `concept_type`

`eval.py`'s metric functions only ever return the *mean* across all trees. `eval.per_concept_metrics`
(added alongside this notebook) returns the same `conciseness`/`distance_score`/`tree_complexity`/
`unk_rate`/`uncovered_rate`/`exact_rate`/`unique_rate` values `evaluate()` would average, one row per
concept instead -- so they can be grouped by `concept_type` before averaging. `semantic_coverage` is
joined in separately (it comes from the transition-matrix `S`, not the context trees). `uniqueness_entropy`
is a property of the *distribution* of tree-shapes across a population, not of any single tree, so it isn't
a per-concept column -- it's recomputed directly per `(method, k, concept_type)` stratum below instead
(`unique_rate` is its per-concept decomposition and doesn't need this treatment).

In [5]:
rows = []  # per-concept rows, before grouping
uniqueness_rows = []  # per (method, k, concept_type) -- see the markdown note above

for category, file_type in CANDIDATE_LISTS:
    method = f"{category}/{file_type}"
    for k in K_LIST:
        T = drilldown.get_candidate_ids(all_candidates, category, file_type, k)
        trees = eval.build_context_trees(mapped_ids, adj, T, D, id_to_label)
        S = tokenizer.compute_semantic_coverage(A, node_to_idx, T, D)

        sem_cov_by_concept = {c: float(S[-1, node_to_idx[c]]) for c in trees if c in node_to_idx}
        pc = eval.per_concept_metrics(trees, T, D).with_columns(
            pl.lit(method).alias("method"),
            pl.lit(k).alias("k"),
            pl.col("concept").replace_strict(sem_cov_by_concept, default=None, return_dtype=pl.Float64).alias("semantic_coverage"),
        )
        rows.append(pc)

        for ctype, ids in concept_type.group_by("concept_type"):
            ctype = ctype[0]
            stratum_ids = set(ids["id"].to_list())
            stratum_trees = {c: t for c, t in trees.items() if c in stratum_ids}
            uniqueness_rows.append({
                "method": method,
                "k": k,
                "concept_type": ctype,
                "uniqueness_entropy": eval.uniqueness_entropy(stratum_trees),
                "n_concepts": len(stratum_trees),
            })

per_concept_all = pl.concat(rows).join(concept_type, left_on="concept", right_on="id", how="left")
uniqueness_df = pl.DataFrame(uniqueness_rows)
per_concept_all.head()

concept,conciseness,distance_score,tree_complexity,unk_rate,exact_rate,method,k,semantic_coverage,concept_type
str,i64,f64,i64,bool,bool,str,i32,f64,str
"""10001005""",9,0.527778,7,false,false,"""greedy_tree_margin/0.8""",1000,1.0,"""SCT_PRE"""
"""10020007""",5,0.458333,4,true,false,"""greedy_tree_margin/0.8""",1000,0.916667,"""SCT_PRE"""
"""10029008""",6,0.277778,7,true,false,"""greedy_tree_margin/0.8""",1000,0.808333,"""SCT_PRE"""
"""1003502008""",8,0.444444,5,true,false,"""greedy_tree_margin/0.8""",1000,0.983333,"""SCT_PRE"""
"""1003502008+125606003""",10,0.346154,10,true,false,"""greedy_tree_margin/0.8""",1000,0.788889,"""SCT_POST"""


## Stratified metric table -- `group_by(method, k, concept_type)`

In [ ]:
METRIC_COLS = ["conciseness", "distance_score", "tree_complexity", "unk_rate", "uncovered_rate",
               "exact_rate", "unique_rate", "semantic_coverage"]

stratified = (
    per_concept_all.group_by(["method", "k", "concept_type"])
    .agg(
        pl.col("conciseness").mean(),
        pl.col("distance_score").mean(),
        pl.col("tree_complexity").mean(),
        pl.col("unk_rate").cast(pl.Float64).mean(),
        pl.col("uncovered_rate").cast(pl.Float64).mean(),
        pl.col("exact_rate").cast(pl.Float64).mean(),
        pl.col("unique_rate").cast(pl.Float64).mean(),
        pl.col("semantic_coverage").mean(),
        pl.len().alias("n_concepts"),
    )
    .join(uniqueness_df, on=["method", "k", "concept_type"], how="left", suffix="_check")
    .sort(["k", "concept_type", "method"])
)

# sanity check: n_concepts from group_by should match uniqueness_df's independently-counted n_concepts
assert (stratified["n_concepts"] == stratified["n_concepts_check"]).all()
stratified = stratified.drop("n_concepts_check")
stratified

In [ ]:
# unstratified reference: same metrics, no concept_type breakdown -- a sanity check that the
# per-type numbers above are a real decomposition of the number you'd otherwise only ever see.
overall = (
    per_concept_all.group_by(["method", "k"])
    .agg(
        pl.col("conciseness").mean(),
        pl.col("distance_score").mean(),
        pl.col("tree_complexity").mean(),
        pl.col("unk_rate").cast(pl.Float64).mean(),
        pl.col("uncovered_rate").cast(pl.Float64).mean(),
        pl.col("exact_rate").cast(pl.Float64).mean(),
        pl.col("unique_rate").cast(pl.Float64).mean(),
        pl.col("semantic_coverage").mean(),
    )
    .sort(["k", "method"])
)
overall

## Does the advantage hold uniformly across `concept_type`?

`CANDIDATE_LISTS[0]`'s metrics minus `CANDIDATE_LISTS[1]`'s, per `k` and `concept_type` --
positive means the first list is ahead (except `unk_rate`/`uncovered_rate`, where lower is
better).

In [8]:
strong_name = f"{CANDIDATE_LISTS[0][0]}/{CANDIDATE_LISTS[0][1]}"
weak_name = f"{CANDIDATE_LISTS[1][0]}/{CANDIDATE_LISTS[1][1]}"

gap = (
    stratified.filter(pl.col("method") == strong_name)
    .select(["k", "concept_type", "n_concepts", *METRIC_COLS, "uniqueness_entropy"])
    .join(
        stratified.filter(pl.col("method") == weak_name).select(["k", "concept_type", *METRIC_COLS, "uniqueness_entropy"]),
        on=["k", "concept_type"],
        suffix="_baseline",
    )
)
for m in [*METRIC_COLS, "uniqueness_entropy"]:
    gap = gap.with_columns((pl.col(m) - pl.col(f"{m}_baseline")).alias(f"{m}_gap"))

gap_display = gap.select(["k", "concept_type", "n_concepts", *[f"{m}_gap" for m in METRIC_COLS], "uniqueness_entropy_gap"]).sort(["k", "concept_type"])
print(f"{strong_name}  minus  {weak_name}:")
gap_display

greedy_tree_margin/0.8  minus  baseline/highest_degree_dist_1:


k,concept_type,n_concepts,conciseness_gap,distance_score_gap,tree_complexity_gap,unk_rate_gap,exact_rate_gap,semantic_coverage_gap,uniqueness_entropy_gap
i32,str,u32,f64,f64,f64,f64,f64,f64,f64
1000,"""SCT_POST""",26311,0.560032,-0.003963,0.182813,-0.108548,0.0,0.063376,0.011083
1000,"""SCT_PRE""",19839,0.933464,0.050095,0.15152,-0.116185,0.000857,0.130296,0.053413
5000,"""SCT_POST""",26311,0.091103,0.004905,0.052069,-0.049105,0.000874,0.017761,-0.001316
5000,"""SCT_PRE""",19839,0.210242,0.039882,0.192399,-0.051162,0.03604,0.04098,0.017252
8000,"""SCT_POST""",26311,0.010528,0.001777,0.035688,-0.016343,0.009958,0.008752,-0.003376
8000,"""SCT_PRE""",19839,-0.259338,0.054545,-0.049398,-0.010283,0.107062,0.020494,0.012766


## Visualize

In [ ]:
PLOT_METRICS = [*METRIC_COLS, "uniqueness_entropy"]
types = sorted(concept_type["concept_type"].unique().to_list())
methods_list = [f"{c}/{f}" for c, f in CANDIDATE_LISTS]

for k in K_LIST:
    fig, axes = plt.subplots(2, 5, figsize=(22, 8))  # 9 metrics, 1 spare slot hidden below
    k_rows = stratified.filter(pl.col("k") == k)
    for ax, metric in zip(axes.flat, PLOT_METRICS):
        x = range(len(types))
        width = 0.8 / len(methods_list)
        for i, method in enumerate(methods_list):
            vals = [
                k_rows.filter((pl.col("concept_type") == t) & (pl.col("method") == method))[metric].item()
                for t in types
            ]
            ax.bar([xi + i * width for xi in x], vals, width, label=method)
        ax.set_xticks([xi + width * (len(methods_list) - 1) / 2 for xi in x])
        ax.set_xticklabels(types)
        ax.set_title(metric)
    for ax in axes.flat[len(PLOT_METRICS):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(f"k = {k}")
    fig.tight_layout()

plt.show()

## Takeaways

From a real run of this notebook (`greedy_tree_margin/1.0` vs. `baseline/highest_degree`,
`k ∈ {1000, 5000}`, `SCT_PRE`: 19,839 concepts, `SCT_POST`: 26,311 concepts -- comparable
sizes, so differences below aren't just small-sample noise):

- **`semantic_coverage` and `unk_rate`: the advantage is uniform.** `greedy_tree_margin/1.0`
  beats `highest_degree` on both metrics, in both concept types, at both `k` values (e.g.
  `semantic_coverage` gap `+0.095`/`+0.121` at `k=1000`, `+0.028`/`+0.048` at `k=5000` for
  `SCT_POST`/`SCT_PRE` respectively -- same sign everywhere, only the magnitude shifts). This
  is good news for the headline claim: it isn't an artifact of one concept type.
- **`conciseness` is *not* uniform -- it flips sign by type at low `k`.** At `k=1000`,
  `greedy_tree_margin/1.0` is *less* concise than `highest_degree` for `SCT_PRE` concepts
  (gap `-0.17`) while being *more* concise for `SCT_POST` (gap `+0.31`). By `k=5000` both
  types favor `greedy_tree_margin/1.0` again. Any paper claim about `conciseness` should
  either qualify it by `k`, or note this instability explicitly -- it's exactly the kind of
  thing that stays invisible until you stratify.
- **`SCT_POST` (compound, post-coordinated expressions) is structurally harder to
  represent than `SCT_PRE`** at low `k` for the baseline (`unk_rate` far higher, `semantic_coverage`
  far lower), which narrows sharply for both methods by `k=5000` -- worth checking whether
  this is because `SCT_POST` ids are graph nodes only in a derived/compositional sense, before
  reading too much into it.
- Re-run the "Parameters" cell with different `CANDIDATE_LISTS` / `K_LIST` to check whether
  these patterns hold for other method pairs (e.g. `most_children` vs. `highest_degree`, or a
  couple of different `greedy_tree_margin` `lam` values against each other).